In [2]:
!pip install --user transformers datasets evaluate rouge-score nltk pandas matplotlib 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24954 sha256=bc0e80db15206e16fca57d6cf4cc93479f074ffdf4025919c86bff937437cb6b
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import glob
from datasets import load_dataset
import datasets
from datasets import Dataset, concatenate_datasets

In [5]:
!git clone https://github.com/XueFuzhao/InstructionWild.git

Cloning into 'InstructionWild'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (227/227), done.
remote: Total 255 (delta 42), reused 186 (delta 17), pack-reused 0
Receiving objects: 100% (255/255), 18.27 MiB | 10.64 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:
from datasets import load_dataset

dataset = load_dataset('reddit') # Negative Samples
print(f"Train dataset size: {len(dataset['train'])}")

In [42]:
df1 = pd.read_json('InstructionWild/data_v2/user_4.jsonl', lines=True) # Positive Samples
df2 = pd.read_json('InstructionWild/data_v2/user_1.jsonl', lines=True)

In [43]:
df2 = df2.drop('instruction_type', axis=1)
df2 = df2.drop('special_tag', axis=1)
df = pd.concat([df1, df2], axis=0)
df.columns = ['text']
df.reset_index(drop=True)
df['labels'] = 1
dataset2 = Dataset.from_pandas(df)
dataset2 = dataset2.remove_columns(['__index_level_0__'])

In [45]:
dataset = dataset['train'].remove_columns(["body", "author", "normalizedBody", "subreddit", "subreddit_id", "id", "summary"])
dataset = dataset.rename_column("content", "text")
new_column = [0] * len(dataset)
dataset = dataset.add_column("labels", new_column)

In [48]:
dataset3 = concatenate_datasets([dataset, dataset2]).shuffle()

In [49]:
dataset3 = dataset3.train_test_split(test_size=0.1)

In [50]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id="google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:

tokenized_inputs = concatenate_datasets([dataset3["train"], dataset3["test"]]).map(lambda x: tokenizer(x["text"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_source_length = max([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Max source length: {max_source_length}") # Get Longest Token Length

tokenized_targets = concatenate_datasets([dataset3["train"], dataset3["test"]]).map(lambda x: tokenizer(x["labels"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_target_length = max([len(x) for x in tokenized_targets["input_ids"]])
print(f"Max target length: {max_target_length}")

del tokenized_inputs
del tokenized_targets # Save Memory

In [33]:
def preprocess_function(sample, padding="max_length"):
    inputs = [item for item in sample["text"]]

    model_inputs = tokenizer(inputs, max_length=max_source_length, padding=padding, truncation=True)

    labels = tokenizer(text_target=sample["labels"], max_length=max_target_length, padding=padding, truncation=True)
    
    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset3.map(preprocess_function, batched=True, remove_columns=['text', 'labels'])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}") # Tokenising Data

DatasetDict({
    train: Dataset({
        features: ['content', 'labels', 'text', '__index_level_0__'],
        num_rows: 3562911
    })
    test: Dataset({
        features: ['content', 'labels', 'text', '__index_level_0__'],
        num_rows: 395879
    })
})

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model_id="google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

In [ ]:
import evaluate
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize
nltk.download("punkt")

metric = evaluate.load("f1")

def postprocess_text(preds, labels): # Convert output to text
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, average='macro')
    result = {k: round(v * 100, 4) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return result

In [ ]:
from transformers import DataCollatorForSeq2Seq

label_pad_token_id = -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
) # Pad to reach max-len

In [ ]:
from huggingface_hub import HfFolder
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

repository_id = f"{model_id.split('/')[1]}-mahir"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=3,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
) # Trainer for Normal Model

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
tokenizer.save_pretrained(repository_id)
trainer.create_model_card()
trainer.push_to_hub()

In [ ]:
class UpWeightedSeq2SeqTrainer(Seq2SeqTrainer): # Editted trainer with higher loss for positive sample (minority)
    def __init__(self, config=None, data_args=None, *args, **kwargs):
        super().__init__(*args, **kwargs)

        if config is None:
            assert isinstance(self.model, PreTrainedModel), (
                "If no `config` is passed the model to be trained has to be of type `PreTrainedModel`, but is"
                f" {self.model.__class__}"
            )
            self.config = self.model.config
        else:
            self.config = config

        self.data_args = data_args
        self.vocab_size = self.config.tgt_vocab_size if isinstance(self.config, FSMTConfig) else self.config.vocab_size

        if self.args.label_smoothing != 0 or (self.data_args is not None and self.data_args.ignore_pad_token_for_loss):
            assert self.config.pad_token_id is not None, (
                "Make sure that `config.pad_token_id` is correcly defined when ignoring `pad_token` for loss"
                " calculation or doing label smoothing."
            )

        if self.config.pad_token_id is None and self.config.eos_token_id is not None:
            logger.warning(
                f"The `config.pad_token_id` is `None`. Using `config.eos_token_id` = {self.config.eos_token_id} for"
                " padding.."
            )

        if self.args.label_smoothing == 0:
            self.loss_fn = torch.nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id, weight=torch.tensor([35.0, 1.0]))
        else:
            # dynamically import label_smoothed_nll_loss
            from utils import label_smoothed_nll_loss

            self.loss_fn = label_smoothed_nll_loss


In [ ]:
repository_id = f"{model_id.split('/')[1]}-mahir-upweighted"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=3,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = UpWeightedSeq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
tokenizer.save_pretrained(repository_id)
trainer.create_model_card()
trainer.push_to_hub()

In [ ]:
dataset4 = dataset.shuffle().select(range(len(dataset)//5))
dataset5 = concatenate_datasets([dataset4, dataset2]).shuffle()

In [ ]:
dataset5 = dataset5.train_test_split(test_size=0.1)

In [ ]:
tokenized_inputs = concatenate_datasets([dataset5["train"], dataset5["test"]]).map(lambda x: tokenizer(x["text"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_source_length = max([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Max source length: {max_source_length}")

tokenized_targets = concatenate_datasets([dataset5["train"], dataset5["test"]]).map(lambda x: tokenizer(x["labels"], truncation=True), batched=True, remove_columns=['text', 'labels'])
max_target_length = max([len(x) for x in tokenized_targets["input_ids"]])
print(f"Max target length: {max_target_length}")

In [ ]:
tokenized_dataset = dataset5.map(preprocess_function, batched=True, remove_columns=['text', 'label'])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")

In [ ]:
from transformers import DataCollatorForSeq2Seq

label_pad_token_id = -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)

In [ ]:
class DownSampledSeq2SeqTrainer(Seq2SeqTrainer): # Editted trainer with higher loss for negative samples as well as less samples (majority)
    def __init__(self, config=None, data_args=None, *args, **kwargs):
        super().__init__(*args, **kwargs)

        if config is None:
            assert isinstance(self.model, PreTrainedModel), (
                "If no `config` is passed the model to be trained has to be of type `PreTrainedModel`, but is"
                f" {self.model.__class__}"
            )
            self.config = self.model.config
        else:
            self.config = config

        self.data_args = data_args
        self.vocab_size = self.config.tgt_vocab_size if isinstance(self.config, FSMTConfig) else self.config.vocab_size

        if self.args.label_smoothing != 0 or (self.data_args is not None and self.data_args.ignore_pad_token_for_loss):
            assert self.config.pad_token_id is not None, (
                "Make sure that `config.pad_token_id` is correcly defined when ignoring `pad_token` for loss"
                " calculation or doing label smoothing."
            )

        if self.config.pad_token_id is None and self.config.eos_token_id is not None:
            logger.warning(
                f"The `config.pad_token_id` is `None`. Using `config.eos_token_id` = {self.config.eos_token_id} for"
                " padding.."
            )

        if self.args.label_smoothing == 0:
            self.loss_fn = torch.nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id, weight=torch.tensor([1.0, 7.0]))
        else:
            # dynamically import label_smoothed_nll_loss
            from utils import label_smoothed_nll_loss

            self.loss_fn = label_smoothed_nll_loss


In [ ]:
from huggingface_hub import HfFolder
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

repository_id = f"{model_id.split('/')[1]}-mahir-downsample"

training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, 
    learning_rate=3e-4,

    num_train_epochs=3,
    logging_dir=f"{repository_id}/logs",
    logging_strategy="epoch", 
    evaluation_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    hub_token=HfFolder.get_token(),
)

trainer = DownSampledSeq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
tokenizer.save_pretrained(repository_id)
trainer.create_model_card()
trainer.push_to_hub()